# 02-1 — Memory Backends

Demonstrates the two production-grade memory backends and the stateless agent pattern.

| Backend | Module | When to use |
|---|---|---|
| `UnboundedMemory` | `ravi.kernel.memory.unbounded_memory` | In-process, no infra needed |
| `RedisMemory` | `ravi.integrations.memory.redis_memory` | Persist across restarts, multi-turn chat |
| `PostgresMemory` | `ravi.integrations.memory.postgres_memory` | Full SQL persistence + audit trail |

**Prerequisites:** Redis running (`make infra-up` or `docker compose up -d redis`)

---
## 1. `UnboundedMemory` — in-process, no infra needed

In [5]:
from ravi.kernel.memory.unbounded_memory import UnboundedMemory
from ravi.kernel.messages.client_messages import UserMessage, AssistantMessage
from ravi.kernel.messages.content import TextBlock

mem = UnboundedMemory()

# Add messages — all memory methods are async
await mem.add_message(UserMessage(content=[TextBlock(text="Hello, what is 2+2?")]))
await mem.add_message(AssistantMessage(content=[TextBlock(text="2+2 = 4")]))
await mem.add_message(UserMessage(content=[TextBlock(text="And 3+3?")]))

msgs = await mem.get_messages()
print(f"Messages stored: {len(msgs)}")
for m in msgs:
    print(f"  {m.__class__.__name__}: {m.content}")


Messages stored: 3
  UserMessage: ["type='text' text='Hello, what is 2+2?'"]
  AssistantMessage: ["type='text' text='2+2 = 4'"]
  UserMessage: ["type='text' text='And 3+3?'"]


---
## 2. `RedisMemory` — persist across restarts

Uses `REDIS_URL` env var (default: `redis://localhost:6379/0`).
Lifecycle: `connect()` → use → `disconnect()`. There is **no** `close()` method.

In [6]:
from ravi.configs.settings import settings
from ravi.integrations.memory.redis_memory import RedisMemory
from ravi.kernel.messages.client_messages import UserMessage, AssistantMessage
from ravi.kernel.messages.content import TextBlock

REDIS_URL = settings.REDIS_URL
SESSION_ID = "demo-session-001"

mem = RedisMemory(session_id=SESSION_ID, redis_url=REDIS_URL)
await mem.connect()

# Restore any previously persisted messages
await mem.restore()
before = await mem.get_messages()
print(f"Messages from previous session: {len(before)}")

# Add new messages
await mem.add_message(UserMessage(content=[TextBlock(text="Tell me a joke")]))
await mem.add_message(AssistantMessage(content=[TextBlock(text="Why did the AI cross the road? To get to the other token!")]))

after = await mem.get_messages()
print(f"Messages after adding: {len(after)}")

# Clear this session
await mem.clear()
empty = await mem.get_messages()
print(f"After clear: {len(empty)} messages")

await mem.disconnect()

Messages from previous session: 0
Messages after adding: 2
After clear: 0 messages


---
## 3. Sliding window via `get_messages(limit=N)`

All memory backends expose `get_messages(limit=N)` to retrieve only the most recent N messages — preventing context overflow without a separate class.


In [8]:
from ravi.kernel.memory.unbounded_memory import UnboundedMemory
from ravi.kernel.messages.client_messages import UserMessage, AssistantMessage
from ravi.kernel.messages.content import TextBlock

# UnboundedMemory supports a sliding-window view via get_messages(limit=N)
mem = UnboundedMemory()

for i in range(6):
    await mem.add_message(UserMessage(content=[TextBlock(text=f"Turn {i+1} user")]))
    await mem.add_message(AssistantMessage(content=[TextBlock(text=f"Turn {i+1} reply")]))

total = await mem.size()
window = await mem.get_messages(limit=4)
print(f"Total stored: {total} messages")
print(f"Window (last 4): {len(window)} messages")
for m in window:
    print(f"  {m.__class__.__name__}: {m.content}")


Total stored: 12 messages
Window (last 4): 4 messages
  UserMessage: ["type='text' text='Turn 5 user'"]
  AssistantMessage: ["type='text' text='Turn 5 reply'"]
  UserMessage: ["type='text' text='Turn 6 user'"]
  AssistantMessage: ["type='text' text='Turn 6 reply'"]


---
## 4. Stateless agent with `RedisMemory`

The agent is created fresh each request — no state stored on the Python object.  
All history lives in Redis, keyed by `session_id`.

In [ ]:
from ravi.configs.settings import settings
from ravi.integrations.memory.redis_memory import RedisMemory
from ravi.extensions.context.redis_model_context import RedisModelContext
from ravi.kernel.messages.client_messages import UserMessage
from ravi.kernel.messages.content import TextBlock

REDIS_URL = settings.REDIS_URL

# --- Turn 1: add a message then disconnect ---
mem1 = RedisMemory(session_id="stateless-demo", redis_url=REDIS_URL)
await mem1.connect()

ctx1 = RedisModelContext(redis_memory=mem1, recent_n=10)

await mem1.add_message(UserMessage(content=[TextBlock(text="My name is Alice")]))
await mem1.disconnect()
print("Turn 1 complete, disconnected")

# --- Turn 2: fresh objects, same session_id — picks up where we left off ---
mem2 = RedisMemory(session_id="stateless-demo", redis_url=REDIS_URL)
await mem2.connect()
await mem2.restore()   # pull history back from Redis

msgs = await mem2.get_messages()
print(f"Turn 2 sees {len(msgs)} message(s) from Turn 1:")
for m in msgs:
    print(f"  {m.__class__.__name__}: {m.content}")

# Clean up
await mem2.clear()
await mem2.disconnect()
print("Cleaned up")

Turn 1 complete, disconnected
Turn 2 sees 1 message(s) from Turn 1:
  UserMessage: ["type='text' text='My name is Alice'"]
Cleaned up


: 

---
## 5. Memory rules — quick reference

```python
# ✅ Always await every memory method
await mem.add_message(msg)
msgs = await mem.get_messages()
msgs_recent = await mem.get_messages(limit=10)  # sliding window built-in
await mem.clear()
count = await mem.size()

# ✅ Redis lifecycle
await mem.connect()     # open connection
await mem.restore()     # reload persisted history
# ... use ...
await mem.disconnect()  # ← correct name, NOT .close()

# ✅ RedisModelContext — stateless agent (ignores in-process memory)
from ravi.extensions.context.redis_model_context import RedisModelContext
ctx = RedisModelContext(redis_memory=mem, recent_n=10)  # ← pass RedisMemory instance

# ✅ Correct import paths
from ravi.kernel.memory.unbounded_memory import UnboundedMemory
from ravi.integrations.memory.redis_memory import RedisMemory       # ← integrations, not core
from ravi.integrations.memory.postgres_memory import PostgresMemory # ← integrations, not core
```
